# TCN 모델 테스트 하는 노트북

In [30]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error
import joblib

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

from tcn import TCN

# 한글 폰트 설정
try:
    plt.rc('font', family='AppleGothic')
except:
    plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [31]:
# ===================================================================
# PART 1: 기본 설정 및 하이퍼파라미터
# ===================================================================
IMAGE_DIRECTORY = '../../data/interim/satellites/gang_nam_rot_90'
TABULAR_DATA_PATH = '../../data/interim/gang_nam_sendimental_score_with_sale.csv'
TIMESTEPS = 36
PCA_COMPONENTS = 64
BATCH_SIZE = 64
EPOCHS = 100
FORECAST_HORIZON = 12 # 미래 예측 기간 (12개월)

In [32]:

# ===================================================================
# PART 2: 데이터 로딩 및 특징 추출
# ===================================================================
print("PART 2: 데이터 로딩 및 특징 추출 시작...")
df = pd.read_csv(TABULAR_DATA_PATH)
df['계약일자'] = pd.to_datetime(df['계약일자'])
df = df.sort_values('계약일자').reset_index(drop=True)
df.drop(columns=['sentiment_score'], axis=1, inplace=True, errors='ignore')
df['image_filename'] = [f'apt_image_{i}.jpg' for i in df.index]
df['image_path'] = df['image_filename'].apply(lambda filename: os.path.join(IMAGE_DIRECTORY, filename))


PART 2: 데이터 로딩 및 특징 추출 시작...


In [33]:
# --- 이미지 특징 추출 함수 ---
def extract_and_reduce_image_features(image_paths, n_components=PCA_COMPONENTS, batch_size=BATCH_SIZE):
    image_input = Input(shape=(224, 224, 3), name='image_input')
    base_cnn = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_tensor=image_input)
    base_cnn.trainable = False
    extractor_model = Model(inputs=image_input, outputs=base_cnn.output, name='resnet50_extractor')
    def image_generator(paths, b_size):
        for i in range(0, len(paths), b_size):
            batch_paths = paths[i:i+b_size]
            batch_images = []
            for path in batch_paths:
                try:
                    img = load_img(path, target_size=(224, 224))
                    img_array = img_to_array(img)
                    batch_images.append(img_array)
                except (FileNotFoundError, IOError):
                    batch_images.append(np.zeros((224, 224, 3)))
            yield preprocess_input(np.array(batch_images))
    gen = image_generator(image_paths, batch_size)
    num_batches = int(np.ceil(len(image_paths) / batch_size))
    features_2048d = extractor_model.predict(gen, steps=num_batches, verbose=1)
    pca = PCA(n_components=n_components)
    features_reduced = pca.fit_transform(features_2048d)
    print(f"PCA 설명된 분산: {np.sum(pca.explained_variance_ratio_):.4f}")
    return features_reduced, pca


image_features_64d, pca_model = extract_and_reduce_image_features(df['image_path'])

image_feature_cols = [f'img_pca_{i}' for i in range(PCA_COMPONENTS)]
df_image_features = pd.DataFrame(image_features_64d, columns=image_feature_cols, index=df.index)
df = pd.concat([df, df_image_features], axis=1)
print("PART 2: 완료!")


337/337 [==============================] - 457s 1s/step
PCA 설명된 분산: 0.9311
PART 2: 완료!


In [34]:
# ===================================================================
# PART 3: 통계 기반 시나리오 설정
# ===================================================================
print("\nPART 3: 과거 데이터 기반 시나리오 값 설정...")
tabular_cols_to_use = [
    '전용면적(㎡)', '층', '건축년도', '아파트 나이', 'leading_index', '건설기성액(백만원)', 
    '부동산_소비심리지수', '주택시장_소비심리지수', '강남구_변동률', '강남구_누계', 
    '토지시장_소비심리지수', '아파트_호수', '아파트_면적', 'rate'
]
final_feature_cols = tabular_cols_to_use + image_feature_cols
TARGET_COL = '면적당 단가(만원)'

X = df[final_feature_cols]
y = df[[TARGET_COL]].values

economic_indicators_for_scenario = [
    'leading_index', '건설기성액(백만원)', '부동산_소비심리지수', '주택시장_소비심리지수', 
    '강남구_변동률', '강남구_누계', '토지시장_소비심리지수', 'rate'
]
df_monthly = df.set_index('계약일자')[economic_indicators_for_scenario].resample('M').mean()
monthly_pct_change = df_monthly.pct_change().dropna()
stats = {'mean': monthly_pct_change.mean(), 'std': monthly_pct_change.std()}
scenarios = {'낙관적': {}, '중립적': {}, '보수적': {}}
for indicator in economic_indicators_for_scenario:
    mean_change, std_change = stats['mean'][indicator], stats['std'][indicator]
    scenarios['중립적'][indicator] = 1 + mean_change
    scenarios['낙관적'][indicator] = 1 + (mean_change + 0.5 * std_change)
    scenarios['보수적'][indicator] = 1 + (mean_change - 0.5 * std_change)
print("통계 기반 시나리오 변동률(월별 곱셈값):")
print(pd.DataFrame(scenarios).round(4))
print("PART 3: 완료!")


PART 3: 과거 데이터 기반 시나리오 값 설정...
통계 기반 시나리오 변동률(월별 곱셈값):
                  낙관적     중립적     보수적
leading_index  1.0041  1.0028  1.0015
건설기성액(백만원)     1.0921  1.0149  0.9378
부동산_소비심리지수     1.0343  1.0035  0.9727
주택시장_소비심리지수    1.0366  1.0039  0.9712
강남구_변동률        1.2187  1.0446  0.8705
강남구_누계         1.4357  1.2159  0.9962
토지시장_소비심리지수    1.0230  1.0010  0.9789
rate           1.0673  1.0121  0.9570
PART 3: 완료!


/var/folders/sq/2pgmkw912zj1zpn9tz6pjdmr0000gn/T/ipykernel_2700/2826426062.py:20: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly = df.set_index('계약일자')[economic_indicators_for_scenario].resample('M').mean()


In [35]:
# ===================================================================
# PART 4: 모델 학습
# ===================================================================
print("\nPART 4: 모델 학습 시작...")
def create_sequences(X_data, y_data, timesteps=TIMESTEPS):
    X, y = [], []
    # 1,3,6개월 예측을 위해 y값 3개 추출. -5는 6개월 후 데이터를 위함.
    for i in range(len(X_data) - timesteps - 5):
        X.append(X_data[i:(i + timesteps)])
        y.append([y_data[i + timesteps], y_data[i + timesteps + 2], y_data[i + timesteps + 5]])
    return np.array(X), np.squeeze(np.array(y), axis=2)

test_size = int(len(df) * 0.1)
val_size = int(len(df) * 0.1)
train_df = df[:-test_size-val_size]
val_df = df[-test_size-val_size:-test_size]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(train_df[final_feature_cols])
y_train_scaled = scaler_y.fit_transform(train_df[[TARGET_COL]])
X_val_scaled = scaler_X.transform(val_df[final_feature_cols])
y_val_scaled = scaler_y.transform(val_df[[TARGET_COL]])

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val_scaled)

final_model = Sequential([
    TCN(input_shape=(TIMESTEPS, X_train_seq.shape[2]), nb_filters=64, kernel_size=3,
        dilations=[1, 2, 4, 8, 16], use_skip_connections=True, dropout_rate=0.1, return_sequences=False),
    Dense(32, activation='relu'),
    Dense(3) # 1,3,6개월 예측을 위해 출력 3개
])
final_model.compile(optimizer='adam', loss='mae')
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
final_model.fit(X_train_seq, y_train_seq, validation_data=(X_val_seq, y_val_seq),
                epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1, callbacks=[early_stopping])
print("PART 4: 완료!")


PART 4: 모델 학습 시작...
Epoch 1/100
269/269 [==============================] - 6s 19ms/step - loss: 0.1960 - val_loss: 0.0907
Epoch 2/100
269/269 [==============================] - 5s 18ms/step - loss: 0.0932 - val_loss: 0.0935
Epoch 3/100
269/269 [==============================] - 5s 18ms/step - loss: 0.0878 - val_loss: 0.0939
Epoch 4/100
269/269 [==============================] - 5s 19ms/step - loss: 0.0863 - val_loss: 0.0894
Epoch 5/100
269/269 [==============================] - 5s 20ms/step - loss: 0.0855 - val_loss: 0.0854
Epoch 6/100
269/269 [==============================] - 5s 18ms/step - loss: 0.0852 - val_loss: 0.0868
Epoch 7/100
269/269 [==============================] - 5s 17ms/step - loss: 0.0846 - val_loss: 0.0872
Epoch 8/100
269/269 [==============================] - 5s 18ms/step - loss: 0.0845 - val_loss: 0.0893
Epoch 9/100
269/269 [==============================] - 5s 18ms/step - loss: 0.0843 - val_loss: 0.0878
Epoch 10/100
269/269 [==============================] - 5s 19

In [ ]:
# ===================================================================
# PART 5: 시나리오 기반 미래 예측 (경고 수정)
# ===================================================================
print("\nPART 5: 시나리오 기반 미래 예측 시작...")
last_sequence_data = X.tail(TIMESTEPS).values
scenario_predictions = {name: [] for name in scenarios.keys()}
current_inputs = {name: last_sequence_data.copy() for name in scenarios.keys()}

for i in tqdm(range(FORECAST_HORIZON), desc="미래 예측 중"):
    for name, params in scenarios.items():
        current_inputs_df = pd.DataFrame(current_inputs[name], columns=final_feature_cols)
        
        current_input_scaled = scaler_X.transform(current_inputs_df)
        
        current_input_reshaped = np.expand_dims(current_input_scaled, axis=0)
        prediction_scaled = final_model.predict(current_input_reshaped, verbose=0)
        prediction_original = scaler_y.inverse_transform(prediction_scaled)
        scenario_predictions[name].append(prediction_original[0][0])
        
        next_feature_row = current_inputs[name][-1, :].copy()
        for indicator in economic_indicators_for_scenario:
            col_idx = X.columns.get_loc(indicator)
            next_feature_row[col_idx] *= params[indicator]
            
        updated_sequence = np.vstack([current_inputs[name][1:], next_feature_row])
        current_inputs[name] = updated_sequence
print("PART 5: 완료!")


PART 5: 시나리오 기반 미래 예측 시작...


미래 예측 중: 100%|██████████| 12/12 [00:00<00:00, 20.59it/s]

PART 5: 완료!


In [41]:

# ===================================================================
# PART 7: 학습된 모델 및 스케일러 저장
# ===================================================================
print("\nPART 7: 학습된 자산(모델, 스케일러) 저장...")
final_model.save('final_tcn_model.h5')
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')
print("PART 7: 완료! 'final_tcn_model.h5', 'scaler_X.pkl', 'scaler_y.pkl' 파일이 저장되었습니다.")


PART 7: 학습된 자산(모델, 스케일러) 저장...
PART 7: 완료! 'final_tcn_model.h5', 'scaler_X.pkl', 'scaler_y.pkl' 파일이 저장되었습니다.
